# Lab 2: The Mirage (LLM vs Physics)

**Estimated Time:** 20 minutes

### 🎯 Learning Objectives:
* Experience AI hallucination in hardware design.
* Implement a cycle-accurate architectural proxy (SCALE-Sim) to enforce the Lighthouse Request.


## Step 1: Prompting the AI Architect
LLMs will confidently generate hardware descriptions that look perfect but violate the laws of physics.

🛠️ **YOUR TURN:** Edit the `prompt` string below. Try to "Prompt Engineer" the LLM into giving a valid hardware dimension that uses exactly 1024 PEs (like 32x32).


In [ ]:
import json, urllib.request

def ask_llm(prompt):
    data = json.dumps({"model": "gemma3:1b", "prompt": prompt, "stream": False, "format": "json"}).encode('utf-8')
    try:
        req = urllib.request.Request("http://localhost:11434/api/generate", data=data, headers={'Content-Type': 'application/json'})
        response = urllib.request.urlopen(req)
        return json.loads(json.loads(response.read())['response'])
    except Exception as e:
        print("Ollama failed, using mock data.")
        return [{'array_rows': 2, 'array_cols': 512}]

# EDIT THIS PROMPT:
prompt = "You are an AI architect. Propose two configurations for a systolic array using exactly 1024 PEs. Output a JSON array with 'array_rows' and 'array_cols'."
llm_candidates = ask_llm(prompt)
print("AI Generated Candidates:", llm_candidates)


## Step 2: The Reality Check (SCALE-Sim)
The LLM output looks plausible. Let's run it through SCALE-Sim physics to see if it meets the 90,000 cycle deadline.


In [ ]:
import sys
from pathlib import Path
import yaml
root_dir = Path().resolve().parent.parent
sys.path.insert(0, str(root_dir / 'labs'))
from arch2_labs.scale_env import run_example

# Inject LLM candidates into the YAML
yaml_path = root_dir / 'labs' / 'examples' / 'scale_proxy_mirage' / 'configs' / 'candidates.yaml'
with open(yaml_path, 'r') as f:
    spec = yaml.safe_load(f)
spec['candidates'] = [{'candidate_id': 'balanced_16x16', 'array_rows': 16, 'array_cols': 16, 'source': 'Human Baseline'}]
for i, c in enumerate(llm_candidates):
    spec['candidates'].append({'candidate_id': f'llm_gen_{i}', 'array_rows': c['array_rows'], 'array_cols': c['array_cols'], 'source': 'Ollama'})
with open(yaml_path, 'w') as f:
    yaml.dump(spec, f)

print("Running SCALE-Sim Verification...")
out_dir = run_example()

# Save out_dir for the next labs so we don't have to re-run SCALE-Sim
with open('.arch2_workshop_state.json', 'w') as f:
    json.dump({"run_archive": str(out_dir)}, f)

with open(out_dir / "verification_evidence.json", 'r') as f:
    evidence = json.load(f)

print("\n--- VERIFICATION RESULTS ---")
for outcome in evidence['candidate_outcomes']:
    status = "✅ PASSED" if outcome['accepted'] else "❌ FAILED"
    print(f"{outcome['candidate_id']}: {status}")
    if not outcome['accepted']:
        print(f"   Reason: {outcome['reason']}")


🗣️ **Discussion Question:** Did prompt engineering guarantee physical success? Why or why not?

## 📚 Further Reading
* **Chapter 5 & 6**: Navigating the Mirage and building resilient Verification Environments.
